# ESBM

## SBMModel

In [1]:
from collections.abc import Callable
from dataclasses import dataclass

import numpy as np
from numpy.typing import NDArray
from scipy.special import betaln


FloatArray = NDArray[np.float64]
IntArray = NDArray[np.int64]


@dataclass(frozen=True)
class SBMModel:
    """
    SBM bayesiano para redes no dirigidas, binarias y sin lazos.

    Modelo:
        Y_ij | z, theta ~ Bernoulli(theta[z_i, z_j]), i < j
        theta_hk ~ Beta(a, b)

    Las probabilidades theta se integran analíticamente.

    Parameters
    ----------
    adjacency : array-like
        Matriz de adyacencia simétrica, binaria y con diagonal nula.
    a, b : float
        Parámetros positivos del prior Beta sobre las probabilidades
        de conexión entre bloques.
    """

    adjacency: NDArray
    a: float = 1.0
    b: float = 1.0

    def __post_init__(self) -> None:
        # Crear una copia para evitar modificar los datos originales.
        Y = np.array(self.adjacency, dtype=np.float64, copy=True)

        if Y.ndim != 2 or Y.shape[0] != Y.shape[1]:
            raise ValueError("La matriz de adyacencia debe ser cuadrada.")

        if Y.shape[0] < 2:
            raise ValueError("La red debe contener al menos dos nodos.")

        if not np.all(np.isfinite(Y)):
            raise ValueError("La matriz contiene valores no finitos.")

        if not np.allclose(Y, Y.T):
            raise ValueError("La matriz debe ser simétrica.")

        if not np.all((Y == 0) | (Y == 1)):
            raise ValueError("La matriz debe contener únicamente 0 y 1.")

        if not np.all(np.diag(Y) == 0):
            raise ValueError("No se permiten lazos: la diagonal debe ser cero.")

        if not np.isfinite(self.a) or not np.isfinite(self.b):
            raise ValueError("Los parámetros a y b deben ser finitos.")

        if self.a <= 0 or self.b <= 0:
            raise ValueError("Los parámetros a y b deben ser positivos.")

        Y.setflags(write=False)
        object.__setattr__(self, "adjacency", Y)
        object.__setattr__(self, "a", float(self.a))
        object.__setattr__(self, "b", float(self.b))

    @property
    def n_nodes(self) -> int:
        """Número de nodos de la red."""
        return self.adjacency.shape[0]

    @staticmethod
    def _canonicalize_partition(z: NDArray) -> IntArray:
        """
        Convierte las etiquetas a enteros consecutivos desde cero.

        Ejemplo:
            [5, 5, 9, 9, 5] -> [0, 0, 1, 1, 0]
        """
        z = np.asarray(z)

        if z.ndim != 1:
            raise ValueError("La partición debe ser un vector unidimensional.")

        if not np.issubdtype(z.dtype, np.integer):
            raise ValueError("Las etiquetas deben ser enteros.")

        if z.size == 0:
            raise ValueError("La partición no puede estar vacía.")

        _, inverse = np.unique(z, return_inverse=True)
        return inverse.astype(np.int64)

    def _validate_partition(self, z: NDArray) -> IntArray:
        """Valida la partición respecto al número de nodos."""
        z = self._canonicalize_partition(z)

        if len(z) != self.n_nodes:
            raise ValueError(
                f"Se esperaban {self.n_nodes} etiquetas; "
                f"se recibieron {len(z)}."
            )

        return z

    def block_statistics(self, z: NDArray) -> dict[str, NDArray]:
        """
        Calcula las estadísticas suficientes por bloque.

        Returns
        -------
        dict
            labels: etiquetas canónicas.
            sizes: tamaños de los bloques.
            edges: número de aristas por bloque.
            non_edges: número de pares sin arista por bloque.

        Cada par no ordenado se cuenta una sola vez.
        """
        z = self._validate_partition(z)

        n_blocks = int(z.max()) + 1
        sizes = np.bincount(z, minlength=n_blocks)

        # Contar aristas entre bloques.
        edges = np.zeros((n_blocks, n_blocks), dtype=np.float64)

        rows, cols = np.triu_indices(self.n_nodes, k=1)

        block_i = np.minimum(z[rows], z[cols])
        block_j = np.maximum(z[rows], z[cols])

        edges = np.zeros((n_blocks, n_blocks), dtype=np.float64)

        np.add.at(edges, (block_i, block_j), self.adjacency[rows, cols])

        edges = edges + np.triu(edges, k=1).T

        # Número total de pares posibles entre bloques.
        possible = np.outer(sizes, sizes).astype(np.float64)
        diagonal = np.diag_indices(n_blocks)
        possible[diagonal] = sizes * (sizes - 1) / 2

        non_edges = possible - edges

        return {
            "labels": z,
            "sizes": sizes,
            "edges": edges,
            "non_edges": non_edges,
        }

    def log_marginal_likelihood(self, z: NDArray) -> float:
        """
        Calcula log p(Y | z), integrando las probabilidades theta.

        log p(Y | z) =
            sum_{h <= k} [
                log B(a + edges_hk, b + non_edges_hk)
                - log B(a, b)
            ]
        """
        stats = self.block_statistics(z)
        edges = stats["edges"]
        non_edges = stats["non_edges"]

        upper = np.triu_indices_from(edges)

        log_prob = (
            betaln(self.a + edges[upper],
                   self.b + non_edges[upper])
            - betaln(self.a, self.b)
        )

        return float(np.sum(log_prob))

    def log_posterior(
        self,
        z: NDArray,
        log_prior: Callable[[IntArray], float] | None = None,
    ) -> float:
        """
        Calcula la log-posterior de la partición, salvo constante:

            log p(z | Y) = log p(Y | z) + log p(z) + C.

        Si log_prior es None, devuelve únicamente log p(Y | z).
        Si se proporciona, debe devolver log p(z) para la partición.
        """
        z = self._validate_partition(z)
        value = self.log_marginal_likelihood(z)

        if log_prior is not None:
            prior_value = float(log_prior(z.copy()))

            if not np.isfinite(prior_value):
                return prior_value

            value += prior_value

        return float(value)

In [2]:
Y = np.array([
    [0, 1, 1, 0, 0, 0],
    [1, 0, 1, 0, 0, 0],
    [1, 1, 0, 1, 0, 0],
    [0, 0, 1, 0, 1, 1],
    [0, 0, 0, 1, 0, 1],
    [0, 0, 0, 1, 1, 0],
])

z = np.array([0, 0, 0, 1, 1, 1])

model = SBMModel(adjacency=Y, a=1.0, b=1.0)

print("Número de nodos:", model.n_nodes)
print("Estadísticas de bloques:")
print(model.block_statistics(z))
print("Log-verosimilitud marginal:", model.log_marginal_likelihood(z))
print("Log-posterior sin prior:", model.log_posterior(z))

Número de nodos: 6
Estadísticas de bloques:
{'labels': array([0, 0, 0, 1, 1, 1]), 'sizes': array([3, 3]), 'edges': array([[3., 1.],
       [1., 3.]]), 'non_edges': array([[0., 8.],
       [8., 0.]])}
Log-verosimilitud marginal: -7.272398392570047
Log-posterior sin prior: -7.272398392570047


## PartitionPrior

In [3]:
from dataclasses import dataclass
from typing import Literal

import numpy as np
from numpy.typing import NDArray
from scipy.special import gammaln, logsumexp


IntArray = NDArray[np.int64]
FloatArray = NDArray[np.float64]

PriorName = Literal["DP", "PY", "DM", "GN", "MFM"]


@dataclass(frozen=True)
class PartitionPrior:
    """
    Distribuciones previas sobre particiones de nodos.

    Priors disponibles
    ------------------
    DP  : Dirichlet Process.
    PY  : Pitman-Yor Process.
    DM  : Dirichlet-Multinomial con H_total categorías.
    GN  : Esquema de urna de Gnedin.
    MFM : Mixture of Finite Mixtures.

    Parámetros
    ----------
    kind : {"DP", "PY", "DM", "GN", "MFM"}
        Tipo de prior.
    alpha : float, optional
        Parámetro de concentración para DP y PY.
    sigma : float, optional
        Parámetro de descuento para PY.
    beta : float, optional
        Parámetro por categoría para DM.
    H_total : int, optional
        Número máximo de categorías para DM.
    gamma : float, optional
        Parámetro del esquema GN.
    rho : float, optional
        Parámetro Dirichlet simétrico para MFM.
    p_k : tuple[float, ...], optional
        Probabilidades P(K=m), para m=1,...,M.
        Se normalizan automáticamente.
    """

    kind: PriorName
    alpha: float | None = None
    sigma: float | None = None
    beta: float | None = None
    H_total: int | None = None
    gamma: float | None = None
    rho: float | None = None
    p_k: tuple[float, ...] | None = None

    def __post_init__(self) -> None:
        kind = self.kind.upper()

        if kind not in {"DP", "PY", "DM", "GN", "MFM"}:
            raise ValueError("kind debe ser DP, PY, DM, GN o MFM.")

        object.__setattr__(self, "kind", kind)

        if kind in {"DP", "PY"}:
            if self.alpha is None or not np.isfinite(self.alpha):
                raise ValueError("Se requiere alpha finito.")

            if kind == "DP" and self.alpha <= 0:
                raise ValueError("Para DP, alpha debe ser positivo.")

        if kind == "PY":
            if self.sigma is None or not np.isfinite(self.sigma):
                raise ValueError("PY requiere sigma finito.")

            if not 0 <= self.sigma < 1:
                raise ValueError("PY requiere 0 <= sigma < 1.")

            if self.alpha <= -self.sigma:
                raise ValueError("PY requiere alpha > -sigma.")

        if kind == "DM":
            if self.beta is None or not np.isfinite(self.beta) or self.beta <= 0:
                raise ValueError("DM requiere beta positivo y finito.")

            if (
                self.H_total is None
                or not isinstance(self.H_total, (int, np.integer))
                or self.H_total < 1
            ):
                raise ValueError("DM requiere H_total entero positivo.")

        if kind == "GN":
            if self.gamma is None or not np.isfinite(self.gamma):
                raise ValueError("GN requiere gamma finito.")

            if not 0 < self.gamma < 1:
                raise ValueError("GN requiere 0 < gamma < 1.")

        if kind == "MFM":
            if self.rho is None or not np.isfinite(self.rho) or self.rho <= 0:
                raise ValueError("MFM requiere rho positivo y finito.")

            if self.p_k is None:
                raise ValueError("MFM requiere especificar p_k.")

            p_k = np.asarray(self.p_k, dtype=float)

            if (
                p_k.ndim != 1
                or p_k.size == 0
                or np.any(~np.isfinite(p_k))
                or np.any(p_k < 0)
                or p_k.sum() <= 0
            ):
                raise ValueError(
                    "p_k debe ser un vector de probabilidades no negativas "
                    "con suma positiva."
                )

            p_k = p_k / p_k.sum()
            object.__setattr__(self, "p_k", tuple(p_k))

    @staticmethod
    def _canonicalize(z: NDArray) -> IntArray:
        """Renombra las comunidades según su primera aparición."""
        z = np.asarray(z)

        if z.ndim != 1 or z.size == 0:
            raise ValueError("z debe ser un vector no vacío.")

        if not np.issubdtype(z.dtype, np.integer):
            raise ValueError("Las etiquetas deben ser enteros.")

        mapping = {}
        canonical = np.empty(len(z), dtype=np.int64)

        for i, label in enumerate(z):
            label = int(label)

            if label not in mapping:
                mapping[label] = len(mapping)

            canonical[i] = mapping[label]

        return canonical

    @staticmethod
    def _sizes(z: IntArray) -> IntArray:
        """Devuelve los tamaños de las comunidades no vacías."""
        return np.bincount(z)

    def _log_V(self, n: int, k: int) -> float:
        """
        Calcula log V_n(k) para MFM mediante log-sum-exp.

        p_k[m-1] = P(K=m), para m=1,...,M.
        """
        if self.kind != "MFM":
            raise ValueError("_log_V solo está disponible para MFM.")

        if k < 1 or n < k:
            return -np.inf

        p_k = np.asarray(self.p_k, dtype=float)
        m_all = np.arange(1, len(p_k) + 1)

        valid = (m_all >= k) & (p_k > 0)

        if not np.any(valid):
            return -np.inf

        m = m_all[valid].astype(float)
        probs = p_k[valid]

        log_terms = (
            gammaln(m + 1)
            - gammaln(m - k + 1)
            + gammaln(m * self.rho)
            - gammaln(m * self.rho + n)
            + np.log(probs)
        )

        return float(logsumexp(log_terms))

    def log_probability(self, z: NDArray) -> float:
        """
        Log-probabilidad previa de una partición.

        Para DP, PY, DM y MFM utiliza sus EPPF.
        Para GN utiliza la probabilidad secuencial inducida
        por los pesos de urna.
        """
        z = self._canonicalize(z)
        sizes = self._sizes(z)

        n = len(z)
        K = len(sizes)

        if self.kind == "DP":
            alpha = self.alpha

            return float(
                gammaln(alpha)
                - gammaln(alpha + n)
                + K * np.log(alpha)
                + np.sum(gammaln(sizes))
            )

        if self.kind == "PY":
            alpha = self.alpha
            sigma = self.sigma

            if K == 1:
                log_product = 0.0
            else:
                j = np.arange(1, K, dtype=float)
                terms = alpha + j * sigma

                if np.any(terms <= 0):
                    return -np.inf

                log_product = np.sum(np.log(terms))

            return float(
                log_product
                + gammaln(alpha + 1)
                - gammaln(alpha + n)
                + np.sum(
                    gammaln(sizes - sigma)
                    - gammaln(1 - sigma)
                )
            )

        if self.kind == "DM":
            beta = self.beta
            H = self.H_total

            if K > H:
                return -np.inf

            log_injections = (
                gammaln(H + 1)
                - gammaln(H - K + 1)
            )

            return float(
                log_injections
                + gammaln(H * beta)
                - gammaln(n + H * beta)
                + np.sum(
                    gammaln(sizes + beta)
                    - gammaln(beta)
                )
            )

        if self.kind == "MFM":
            log_v = self._log_V(n, K)

            if not np.isfinite(log_v):
                return -np.inf

            rho = self.rho

            return float(
                log_v
                + np.sum(
                    gammaln(sizes + rho)
                    - gammaln(rho)
                )
            )

        # GN: probabilidad secuencial basada en la urna de referencia.
        log_prob = 0.0
        current_sizes = np.array([1], dtype=np.int64)

        for i in range(1, n):
            weights = self.urn_weights(current_sizes)
            total = weights.sum()
            target = int(z[i])

            if not np.isfinite(total) or total <= 0:
                raise ValueError("Los pesos GN no son normalizables.")

            if target < len(current_sizes):
                weight = weights[target]
            else:
                if target != len(current_sizes):
                    raise ValueError("Partición no canónica.")
                weight = weights[-1]

            if weight <= 0:
                return -np.inf

            log_prob += np.log(weight / total)

            if target == len(current_sizes):
                current_sizes = np.append(current_sizes, 1)
            else:
                current_sizes[target] += 1

        return float(log_prob)

    def urn_weights(self, sizes: NDArray) -> FloatArray:
        """
        Devuelve pesos para comunidades existentes y una nueva.

        El último elemento corresponde a crear una nueva comunidad.
        Los pesos son relativos; predictive_probabilities() los normaliza.
        """
        sizes = np.asarray(sizes, dtype=np.float64)

        if sizes.ndim != 1 or sizes.size == 0 or np.any(sizes <= 0):
            raise ValueError("sizes debe contener tamaños positivos.")

        K = len(sizes)
        n = int(sizes.sum())

        if self.kind == "DP":
            weights = np.append(sizes, self.alpha)

        elif self.kind == "PY":
            weights = np.append(
                sizes - self.sigma,
                self.alpha + K * self.sigma,
            )

        elif self.kind == "DM":
            if K > self.H_total:
                raise ValueError("Hay más comunidades que H_total.")

            weights = np.append(
                sizes + self.beta,
                self.beta * (self.H_total - K),
            )

        elif self.kind == "MFM":
            log_v_n_k = self._log_V(n, K)

            if not np.isfinite(log_v_n_k):
                raise ValueError(
                    "La partición tiene probabilidad previa nula bajo MFM."
                )

            log_v_next_k = self._log_V(n + 1, K)
            log_v_next_k1 = self._log_V(n + 1, K + 1)

            existing_ratio = (
                0.0
                if not np.isfinite(log_v_next_k)
                else np.exp(log_v_next_k - log_v_n_k)
            )

            new_ratio = (
                0.0
                if not np.isfinite(log_v_next_k1)
                else np.exp(log_v_next_k1 - log_v_n_k)
            )

            weights = np.append(
                (sizes + self.rho) * existing_ratio,
                self.rho * new_ratio,
            )

        else:  # GN: pesos del código de referencia de Legramanti.
            gamma = self.gamma

            existing = (sizes + 1) * (n - K + gamma)
            new = K**2 - K * gamma

            weights = np.append(existing, new)

        if np.any(~np.isfinite(weights)) or np.any(weights < 0):
            raise ValueError("El prior produjo pesos inválidos.")

        if weights.sum() <= 0:
            raise ValueError("La suma de los pesos debe ser positiva.")

        return weights.astype(np.float64)

    def predictive_probabilities(self, sizes: NDArray) -> FloatArray:
        """Normaliza los pesos para obtener probabilidades predictivas."""
        weights = self.urn_weights(sizes)
        return weights / weights.sum()

In [22]:
z = np.array([0, 0, 1, 1, 1, 2, 2, 2, 1])
z_renamed = np.array([7, 7, 3, 3, 3, 9, 9, 9, 3])

priors = {
    "DP": PartitionPrior(kind="DP", alpha=1.5),
    "PY": PartitionPrior(kind="PY", alpha=1.0, sigma=0.2),
    "DM": PartitionPrior(kind="DM", beta=1.0, H_total=5),
    "GN": PartitionPrior(kind="GN", gamma=0.4),
    "MFM": PartitionPrior(kind="MFM", rho=1.0, p_k= tuple(0.5 * (1 - 0.5) ** np.arange(10))),
}

for name, prior in priors.items():
    lp = prior.log_probability(z)
    lp_renamed = prior.log_probability(z_renamed)

    assert np.isclose(lp, lp_renamed)
    print(f"{name}: log-prior = {lp:.6f}")

DP: log-prior = -10.360105
PY: log-prior = -10.747272
DM: log-prior = -9.616805
GN: log-prior = -11.191618
MFM: log-prior = -10.846525


In [23]:
sizes = np.array([3, 2, 1])

for name, prior in priors.items():
    probabilities = prior.predictive_probabilities(sizes)

    assert np.all(probabilities >= 0)
    assert np.isclose(probabilities.sum(), 1.0)

    print(
        f"{name}: probabilidades = {np.round(probabilities, 4)}, "
        f"suma = {probabilities.sum():.4f}"
    )

DP: probabilidades = [0.4    0.2667 0.1333 0.2   ], suma = 1.0000
PY: probabilidades = [0.4    0.2571 0.1143 0.2286], suma = 1.0000
DM: probabilidades = [0.3636 0.2727 0.1818 0.1818], suma = 1.0000
GN: probabilidades = [0.3542 0.2656 0.1771 0.2031], suma = 1.0000
MFM: probabilidades = [0.4046 0.3035 0.2023 0.0896], suma = 1.0000


## Sampler

In [24]:
from abc import ABC, abstractmethod
from dataclasses import dataclass

import numpy as np
from numpy.typing import NDArray


@dataclass(frozen=True)
class SamplerConfig:
    n_iterations: int = 5_000
    burn_in: int = 1_000
    thin: int = 5

    def __post_init__(self):
        if self.n_iterations < 1:
            raise ValueError("n_iterations debe ser positivo.")
        if not 0 <= self.burn_in < self.n_iterations:
            raise ValueError("burn_in debe estar entre 0 y n_iterations.")
        if self.thin < 1:
            raise ValueError("thin debe ser positivo.")


class BaseSampler(ABC):
    """Clase base para muestreadores MCMC de particiones SBM."""

    def __init__(
        self,
        model: SBMModel,
        prior: PartitionPrior,
        config: SamplerConfig | None = None,
    ):
        self.model = model
        self.prior = prior
        self.config = config or SamplerConfig()

    @staticmethod
    def _canonicalize(z: NDArray) -> NDArray[np.int64]:
        """Renombra bloques según su primera aparición."""
        z = np.asarray(z)

        if z.ndim != 1 or not np.issubdtype(z.dtype, np.integer):
            raise ValueError("La partición debe ser un vector de enteros.")

        mapping = {}
        result = np.empty(len(z), dtype=np.int64)

        for i, label in enumerate(z):
            label = int(label)
            if label not in mapping:
                mapping[label] = len(mapping)
            result[i] = mapping[label]

        return result

    def log_target(self, z: NDArray) -> float:
        """Log-posterior no normalizada de la partición."""
        return self.model.log_posterior(
            z,
            log_prior=self.prior.log_probability,
        )

    @abstractmethod
    def _step(
        self,
        z: NDArray[np.int64],
        rng: np.random.Generator,
    ) -> NDArray[np.int64]:
        """Ejecuta una transición del algoritmo."""
        raise NotImplementedError

    def fit(
        self,
        initial_partition: NDArray,
        seed: int = 42,
    ) -> dict[str, object]:
        """Ejecuta la cadena y guarda muestras y trazas."""

        z = self._canonicalize(initial_partition)

        if len(z) != self.model.n_nodes:
            raise ValueError("La partición debe tener una etiqueta por nodo.")

        rng = np.random.default_rng(seed)
        samples = []
        log_posterior_trace = []
        n_communities_trace = []

        for iteration in range(self.config.n_iterations):
            z = self._canonicalize(self._step(z.copy(), rng))

            log_posterior_trace.append(self.log_target(z))
            n_communities_trace.append(len(np.unique(z)))

            if (
                iteration >= self.config.burn_in
                and (iteration - self.config.burn_in) % self.config.thin == 0
            ):
                samples.append(z.copy())

        return {
            "samples": np.asarray(samples, dtype=np.int64),
            "log_posterior": np.asarray(log_posterior_trace),
            "n_communities": np.asarray(n_communities_trace),
            "final_partition": z.copy(),
            "seed": seed,
        }

In [25]:
from scipy.special import logsumexp


class GibbsSampler(BaseSampler):
    """Gibbs sampler colapsado con reasignación de un nodo a la vez."""

    def _step(
        self,
        z: NDArray[np.int64],
        rng: np.random.Generator,
    ) -> NDArray[np.int64]:

        n = len(z)

        # Recorrido aleatorio de nodos para evitar un orden fijo.
        for i in rng.permutation(n):
            mask = np.arange(n) != i

            # Retirar el nodo y eliminar bloques vacíos.
            z_minus = self._canonicalize(z[mask])
            n_blocks = len(np.unique(z_minus))

            log_weights = []
            candidates = []

            # Probar todas las comunidades existentes y una nueva.
            for k in range(n_blocks + 1):
                candidate = np.empty(n, dtype=np.int64)
                candidate[mask] = z_minus
                candidate[i] = k

                candidate = self._canonicalize(candidate)

                log_weight = self.log_target(candidate)

                candidates.append(candidate)
                log_weights.append(log_weight)

            log_weights = np.asarray(log_weights, dtype=float)

            if np.all(np.isneginf(log_weights)):
                raise RuntimeError(
                    "Todas las asignaciones candidatas tienen "
                    "probabilidad posterior nula."
                )

            probabilities = np.exp(
                log_weights - logsumexp(log_weights)
            )

            selected = rng.choice(
                len(candidates),
                p=probabilities,
            )

            z = candidates[selected].copy()

        return self._canonicalize(z)

In [26]:
Y

array([[0, 1, 1, 0, 0, 0],
       [1, 0, 1, 0, 0, 0],
       [1, 1, 0, 1, 0, 0],
       [0, 0, 1, 0, 1, 1],
       [0, 0, 0, 1, 0, 1],
       [0, 0, 0, 1, 1, 0]])

In [27]:
model = SBMModel(
    adjacency=Y,
    a=1.0,
    b=1.0,
)

prior = PartitionPrior(
    kind="DP",
    alpha=1.5,
)

config = SamplerConfig(
    n_iterations=1500,
    burn_in=100,
    thin=5,
)

sampler = GibbsSampler(
    model=model,
    prior=prior,
    config=config,
)

result = sampler.fit(
    initial_partition=np.array([0, 0, 0, 1, 1, 1]),
    seed=123,
)

print("Forma de las muestras:", result["samples"].shape)
print("Número final de comunidades:", len(np.unique(result["final_partition"])))
print(result["samples"][-2:])

Forma de las muestras: (280, 6)
Número final de comunidades: 4
[[0 1 0 2 2 3]
 [0 1 0 2 2 3]]


In [33]:
samples = result["samples"]

n_communities = np.array([
    len(np.unique(z))
    for z in samples
])

values, counts = np.unique(
    n_communities,
    return_counts=True,
)

frequencies = counts / counts.sum()

for k, frequency in zip(values, frequencies):
    print(f"K = {k}: frecuencia = {frequency:.3f}")

K = 1: frecuencia = 0.011
K = 2: frecuencia = 0.393
K = 3: frecuencia = 0.407
K = 4: frecuencia = 0.164
K = 5: frecuencia = 0.025


In [41]:
import numpy as np
from itertools import combinations
from scipy.special import betaln, gammaln, logsumexp

Y4 = np.array([
    [0, 1, 1, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 1],
    [0, 1, 1, 0]
], dtype=int)

a = 1.0
b = 1.0
alpha = 1.5

assert Y4.shape == (4, 4)
assert np.array_equal(Y4, Y4.T)
assert np.all(np.diag(Y4) == 0)
assert np.all(np.isin(Y4, [0, 1]))

print("Número de nodos:", len(Y4))
print("Número de aristas:", int(np.triu(Y4, k=1).sum()))


def enumerate_partitions(n):
    """Enumera todas las particiones de n elementos."""
    if n < 1:
        raise ValueError("n debe ser positivo.")

    partitions = []

    def recurse(z, max_label):
        i = len(z)

        if i == n:
            partitions.append(tuple(z))
            return

        for label in range(max_label + 2):
            recurse(z + [label], max(max_label, label))

    recurse([0], 0)
    return partitions


partitions = enumerate_partitions(4)

print("Particiones enumeradas:", len(partitions))
assert len(partitions) == 15
assert len(set(partitions)) == 15


def log_marginal_sbm(Y, z, a=1.0, b=1.0):
    """Log-verosimilitud marginal de un SBM binario no dirigido."""
    Y = np.asarray(Y)
    z = np.asarray(z)

    n = len(z)

    if Y.shape != (n, n):
        raise ValueError("Dimensiones incompatibles entre Y y z.")
    if not np.array_equal(Y, Y.T):
        raise ValueError("Y debe ser simétrica.")
    if np.any(np.diag(Y) != 0):
        raise ValueError("Y debe tener diagonal cero.")
    if not np.all(np.isin(Y, [0, 1])):
        raise ValueError("Y debe ser binaria.")
    if a <= 0 or b <= 0:
        raise ValueError("a y b deben ser positivos.")

    labels = sorted(set(z))
    total = 0.0

    for pos, h in enumerate(labels):
        for k in labels[pos:]:
            nodes_h = np.flatnonzero(z == h)
            nodes_k = np.flatnonzero(z == k)

            if h == k:
                possible = len(nodes_h) * (len(nodes_h) - 1) // 2
                subgraph = Y[np.ix_(nodes_h, nodes_h)]
                edges = int(np.triu(subgraph, k=1).sum())
            else:
                possible = len(nodes_h) * len(nodes_k)
                edges = int(Y[np.ix_(nodes_h, nodes_k)].sum())

            non_edges = possible - edges

            total += (
                betaln(a + edges, b + non_edges)
                - betaln(a, b)
            )

    return float(total)

def log_prior_dp(z, alpha=1.5):
    z = np.asarray(z)
    n = len(z)
    sizes = np.unique(z, return_counts=True)[1]
    K = len(sizes)

    return (
        gammaln(alpha)
        - gammaln(alpha + n)
        + K * np.log(alpha)
        + np.sum(gammaln(sizes))
    )


log_priors = np.array([
    log_prior_dp(z, alpha) for z in partitions
])

prior_probabilities = np.exp(log_priors)

print("Suma de probabilidades del prior:",
      prior_probabilities.sum())

assert np.isclose(prior_probabilities.sum(), 1.0)


log_likelihoods = np.array([
    log_marginal_sbm(Y4, z, a, b)
    for z in partitions
])

log_joint = log_likelihoods + log_priors
log_evidence = logsumexp(log_joint)
exact_probabilities = np.exp(log_joint - log_evidence)

print("Suma posterior:", exact_probabilities.sum())
assert np.isclose(exact_probabilities.sum(), 1.0)

order = np.argsort(exact_probabilities)[::-1]

print("\n Quince particiones con mayor probabilidad posterior:")
for idx in order[:15]:
    print(
        f"z={partitions[idx]}, "
        f"K={len(set(partitions[idx]))}, "
        f"p(z|Y)={exact_probabilities[idx]:.6f}"
    )


model4 = SBMModel(adjacency=Y4, a=a, b=b)
prior4 = PartitionPrior(kind="DP", alpha=alpha)

config4 = SamplerConfig(
    n_iterations=50_000,
    burn_in=5_000,
    thin=5
)

sampler4 = GibbsSampler(
    model=model4,
    prior=prior4,
    config=config4
)

result4 = sampler4.fit(
    initial_partition=np.array([0, 0, 1, 1]),
    seed=123
)

def canonical_partition(z):
    mapping = {}
    result = []

    for label in z:
        if label not in mapping:
            mapping[label] = len(mapping)
        result.append(mapping[label])

    return tuple(result)


samples = result4["samples"]

if isinstance(samples, np.ndarray) and samples.ndim == 2:
    samples = [canonical_partition(z) for z in samples]
else:
    samples = [canonical_partition(z) for z in samples]

exact_map = {
    z: p for z, p in zip(partitions, exact_probabilities)
}

counts = {z: 0 for z in partitions}
for z in samples:
    counts[z] += 1

empirical = np.array([
    counts[z] / len(samples) for z in partitions
])

exact = np.array([
    exact_map[z] for z in partitions
])

tv_distance = 0.5 * np.sum(np.abs(empirical - exact))

print("Número de muestras:", len(samples))
print("Distancia de variación total:", tv_distance)

print("\nComparación de probabilidades:")
for idx in order[:10]:
    print(
        f"{partitions[idx]} | "
        f"exacta={exact[idx]:.4f} | "
        f"Gibbs={empirical[idx]:.4f}"
    )

Número de nodos: 4
Número de aristas: 5
Particiones enumeradas: 15
Suma de probabilidades del prior: 1.0
Suma posterior: 0.9999999999999998

 Quince particiones con mayor probabilidad posterior:
z=(0, 0, 0, 0), K=1, p(z|Y)=0.176783
z=(0, 1, 1, 0), K=2, p(z|Y)=0.092811
z=(0, 1, 2, 0), K=3, p(z|Y)=0.077343
z=(0, 1, 1, 1), K=2, p(z|Y)=0.077343
z=(0, 0, 1, 0), K=2, p(z|Y)=0.077343
z=(0, 1, 0, 0), K=2, p(z|Y)=0.077343
z=(0, 1, 1, 2), K=3, p(z|Y)=0.077343
z=(0, 0, 0, 1), K=2, p(z|Y)=0.077343
z=(0, 1, 2, 3), K=4, p(z|Y)=0.065258
z=(0, 1, 2, 2), K=3, p(z|Y)=0.038671
z=(0, 1, 2, 1), K=3, p(z|Y)=0.038671
z=(0, 1, 0, 2), K=3, p(z|Y)=0.038671
z=(0, 0, 1, 2), K=3, p(z|Y)=0.038671
z=(0, 1, 0, 1), K=2, p(z|Y)=0.023203
z=(0, 0, 1, 1), K=2, p(z|Y)=0.023203
Número de muestras: 9000
Distancia de variación total: 0.018399696153580544

Comparación de probabilidades:
(0, 0, 0, 0) | exacta=0.1768 | Gibbs=0.1812
(0, 1, 1, 0) | exacta=0.0928 | Gibbs=0.0877
(0, 1, 2, 0) | exacta=0.0773 | Gibbs=0.0819
(0, 1, 1, 

In [42]:
import numpy as np

class SplitMergeSampler(BaseSampler):
    """
    MCMC split-merge para particiones de un SBM.

    Si los nodos elegidos están en el mismo bloque:
        propone dividirlo.

    Si están en bloques distintos:
        propone fusionarlos.

    La aceptación incluye la corrección de Metropolis-Hastings.
    """

    def _step(self, z, rng):
        z = self._canonicalize(z.copy())
        n = len(z)

        # Elegir dos nodos distintos.
        i, j = rng.choice(n, size=2, replace=False)

        same_block = z[i] == z[j]

        if same_block:
            # ---------------------------
            # PROPUESTA DE DIVISIÓN
            # ---------------------------
            old_block = z[i]
            members = np.flatnonzero(z == old_block)

            # Los nodos ancla quedan en bloques diferentes.
            candidate = z.copy()
            candidate[j] = z.max() + 1

            # Los demás miembros se asignan aleatoriamente
            # a uno de los dos bloques, con probabilidad 1/2.
            others = members[(members != i) & (members != j)]

            for node in others:
                if rng.random() < 0.5:
                    candidate[node] = candidate[i]
                else:
                    candidate[node] = candidate[j]

            candidate = self._canonicalize(candidate)

            # Probabilidad de la propuesta de división.
            log_q_forward = -len(others) * np.log(2.0)

            # La propuesta inversa es una fusión determinista.
            log_q_reverse = 0.0

        else:
            # ---------------------------
            # PROPUESTA DE FUSIÓN
            # ---------------------------
            block_i = z[i]
            block_j = z[j]

            members_i = np.flatnonzero(z == block_i)
            members_j = np.flatnonzero(z == block_j)

            # Fusionar ambos bloques.
            candidate = z.copy()
            candidate[candidate == block_j] = block_i
            candidate = self._canonicalize(candidate)

            # En la propuesta inversa, los nodos no ancla
            # deben recuperar sus asignaciones originales.
            n_others = len(members_i) + len(members_j) - 2

            log_q_forward = 0.0
            log_q_reverse = -n_others * np.log(2.0)

        # Razón de aceptación de Metropolis-Hastings.
        log_ratio = (
            self.log_target(candidate)
            - self.log_target(z)
            + log_q_reverse
            - log_q_forward
        )

        # Aceptar o rechazar sin exponenciar log_ratio.
        if np.log(rng.random()) < min(0.0, log_ratio):
            return candidate

        return z

In [48]:
split_merge = SplitMergeSampler(
    model=model4,
    prior=prior4,
    config=config4
)

result_sm = split_merge.fit(
    initial_partition=np.array([0, 0, 1, 1]),
    seed=123
)


samples = result_sm["samples"]

if isinstance(samples, np.ndarray) and samples.ndim == 2:
    samples = [canonical_partition(z) for z in samples]
else:
    samples = [canonical_partition(z) for z in samples]

exact_map = {
    z: p for z, p in zip(partitions, exact_probabilities)
}

counts = {z: 0 for z in partitions}
for z in samples:
    counts[z] += 1

empirical = np.array([
    counts[z] / len(samples) for z in partitions
])

exact = np.array([
    exact_map[z] for z in partitions
])

tv_distance = 0.5 * np.sum(np.abs(empirical - exact))

print("Número de muestras:", len(samples))
print("Distancia de variación total:", tv_distance)

print("\nComparación de probabilidades:")
for idx in order[:10]:
    print(
        f"{partitions[idx]} | "
        f"exacta={exact[idx]:.4f} | "
        f"split-merge={empirical[idx]:.4f}"
    )

Número de muestras: 9000
Distancia de variación total: 0.01728388923416892

Comparación de probabilidades:
(0, 0, 0, 0) | exacta=0.1768 | split-merge=0.1777
(0, 1, 1, 0) | exacta=0.0928 | split-merge=0.0887
(0, 1, 2, 0) | exacta=0.0773 | split-merge=0.0744
(0, 1, 1, 1) | exacta=0.0773 | split-merge=0.0792
(0, 0, 1, 0) | exacta=0.0773 | split-merge=0.0798
(0, 1, 0, 0) | exacta=0.0773 | split-merge=0.0781
(0, 1, 1, 2) | exacta=0.0773 | split-merge=0.0732
(0, 0, 0, 1) | exacta=0.0773 | split-merge=0.0801
(0, 1, 2, 3) | exacta=0.0653 | split-merge=0.0659
(0, 1, 2, 2) | exacta=0.0387 | split-merge=0.0364


In [46]:
import numpy as np
from scipy.special import logsumexp


class ChaperonesSampler(BaseSampler):
    """
    Gibbs sampler de Chaperones para particiones de un SBM.

    Selecciona dos nodos guía y actualiza los nodos de los
    bloques que contenían inicialmente a esos guías.

    Los nodos no guía solo pueden pertenecer a un bloque
    que contenga al menos uno de los guías.
    """

    def _valid_state(self, z, anchors, union):
        """Comprueba que todos los nodos de union sigan acompañados."""
        i, j = anchors
        anchor_labels = {z[i], z[j]}

        for node in union:
            if node not in anchors and z[node] not in anchor_labels:
                return False

        return True

    def _step(self, z, rng):
        z = self._canonicalize(np.asarray(z, dtype=np.int64).copy())
        n = len(z)

        if n < 2:
            raise ValueError("Chaperones requiere al menos dos nodos.")

        # 1. Seleccionar dos guías, independientemente del estado.
        i, j = rng.choice(n, size=2, replace=False)
        anchors = (int(i), int(j))

        # Congelar la unión de los bloques originales de los guías.
        initial_labels = {z[i], z[j]}
        union = np.flatnonzero(np.isin(z, list(initial_labels)))

        # 2. Actualizar aleatoriamente los nodos de esa unión.
        for node in rng.permutation(union):
            node = int(node)
            mask = np.arange(n) != node

            # Etiquetas de bloques que siguen existiendo
            # después de retirar temporalmente el nodo.
            labels_in_union = np.unique(z[union[union != node]])

            if node in anchors:
                # Un guía puede cambiar de bloque o crear uno nuevo.
                options = list(labels_in_union)
                options.append(int(z.max()) + 1)
            else:
                # Un hijo solo puede ir al bloque de uno de los guías.
                anchor_labels = {z[i], z[j]}
                options = [
                    label for label in labels_in_union
                    if label in anchor_labels
                ]

            candidates = []
            log_weights = []
            seen = set()

            for label in options:
                candidate = z.copy()
                candidate[node] = label
                candidate = self._canonicalize(candidate)

                # Evitar propuestas duplicadas por canonización.
                key = tuple(candidate)
                if key in seen:
                    continue
                seen.add(key)

                # Mantener la unión original y la restricción
                # de acompañamiento de los nodos no guía.
                if not self._valid_state(candidate, anchors, union):
                    continue

                candidates.append(candidate)
                log_weights.append(self.log_target(candidate))

            if not candidates:
                raise RuntimeError(
                    "No se encontraron asignaciones candidatas válidas."
                )

            log_weights = np.asarray(log_weights, dtype=float)

            if np.all(np.isneginf(log_weights)):
                raise RuntimeError(
                    "Todas las candidatas tienen posterior nula."
                )

            probabilities = np.exp(
                log_weights - logsumexp(log_weights)
            )

            selected = rng.choice(
                len(candidates),
                p=probabilities
            )

            z = candidates[selected].copy()

        return self._canonicalize(z)

In [49]:
chaperones = ChaperonesSampler(
    model=model4,
    prior=prior4,
    config=config4
)

result_ch = chaperones.fit(
    initial_partition=np.array([0, 0, 1, 1]),
    seed=123
)

print("Forma de las muestras:", result_ch["samples"].shape)
print("Número de bloques final:",
      len(np.unique(result_ch["final_partition"])))

samples = result_ch["samples"]

if isinstance(samples, np.ndarray) and samples.ndim == 2:
    samples = [canonical_partition(z) for z in samples]
else:
    samples = [canonical_partition(z) for z in samples]

exact_map = {
    z: p for z, p in zip(partitions, exact_probabilities)
}

counts = {z: 0 for z in partitions}
for z in samples:
    counts[z] += 1

empirical = np.array([
    counts[z] / len(samples) for z in partitions
])

exact = np.array([
    exact_map[z] for z in partitions
])

tv_distance = 0.5 * np.sum(np.abs(empirical - exact))

print("Número de muestras:", len(samples))
print("Distancia de variación total:", tv_distance)

print("\nComparación de probabilidades:")
for idx in order[:10]:
    print(
        f"{partitions[idx]} | "
        f"exacta={exact[idx]:.4f} | "
        f"chaperones={empirical[idx]:.4f}"
    )

Forma de las muestras: (9000, 4)
Número de bloques final: 3
Número de muestras: 9000
Distancia de variación total: 0.010842552309923367

Comparación de probabilidades:
(0, 0, 0, 0) | exacta=0.1768 | chaperones=0.1744
(0, 1, 1, 0) | exacta=0.0928 | chaperones=0.0952
(0, 1, 2, 0) | exacta=0.0773 | chaperones=0.0777
(0, 1, 1, 1) | exacta=0.0773 | chaperones=0.0786
(0, 0, 1, 0) | exacta=0.0773 | chaperones=0.0800
(0, 1, 0, 0) | exacta=0.0773 | chaperones=0.0787
(0, 1, 1, 2) | exacta=0.0773 | chaperones=0.0781
(0, 0, 0, 1) | exacta=0.0773 | chaperones=0.0771
(0, 1, 2, 3) | exacta=0.0653 | chaperones=0.0623
(0, 1, 2, 2) | exacta=0.0387 | chaperones=0.0343


## Simulation

## Applications

### Zachary Karate Club

In [28]:
# Zachary Karate Club network

import networkx as nx
import numpy as np

G = nx.karate_club_graph()

# Matriz de adyacencia
Y = nx.to_numpy_array(G, dtype=np.int64)

# Partición observada: clubes originales
labels = [G.nodes[i]["club"] for i in G.nodes()]

# Convertir etiquetas a enteros consecutivos
_, z_true = np.unique(labels, return_inverse=True)
z_true = z_true.astype(np.int64)

print("Número de nodos:", G.number_of_nodes())
print("Número de aristas:", G.number_of_edges())
print("Número de comunidades reales:", len(np.unique(z_true)))
print("Tamaño de la matriz:", Y.shape)

Número de nodos: 34
Número de aristas: 78
Número de comunidades reales: 2
Tamaño de la matriz: (34, 34)


In [30]:
import pandas as pd

priors = {
    "DP": PartitionPrior(kind="DP", alpha=1.5),

    "PY": PartitionPrior(
        kind="PY", alpha=1.0, sigma=0.25
    ),

    "DM": PartitionPrior(
        kind="DM", beta=1.0, H_total=5
    ),

    "MFM": PartitionPrior(
        kind="MFM",
        rho=1.0,
        p_k=(0.05, 0.10, 0.20, 0.30, 0.35)
    ),

    "GN": PartitionPrior(kind="GN", gamma=0.5),
}

results = []

for name, prior in priors.items():
    results.append({
        "Prior": name,
        "Log-probabilidad de la partición real":
            prior.log_probability(z_true),
        "Comunidades reales": len(np.unique(z_true)),
        "Tamaños de comunidades":
            str(np.bincount(z_true))
    })

comparison = pd.DataFrame(results)
comparison

,Prior,Log-probabilidad de la partición real,Comunidades reales,Tamaños de comunidades
0,DP,-28.321062,2,[17 17]
1,PY,-28.818476,2,[17 17]
2,DM,-29.784266,2,[17 17]
3,MFM,-26.377086,2,[17 17]
4,GN,-26.423400,2,[17 17]


### Political blogs

### Dolphins